In [11]:
def optimize_parameters(data, isotherm_model, error_function, initial_guess):
    """
    Optimiza los parámetros del modelo isotérmico minimizando una función de error.

    Parameters:
        data (dict): {'x': valores independientes, 'y': valores dependientes}.
        isotherm_model (func): Función que calcula valores predichos basado en parámetros.
        error_function (func): Función de error que se minimizará.
        initial_guess (list): Valores iniciales de los parámetros.

    Returns:
        list: Parámetros optimizados.
    """
    from scipy.optimize import minimize

    def objective_function(params):
        y_pred = isotherm_model(data['x'], *params)
        return error_function(data['y'], y_pred)

    result = minimize(objective_function, initial_guess, method="Nelder-Mead")
    return result.x
    
def calculate_all_errors(parameters, isotherm_model, data, error_functions):
    """
    Calcula todas las funciones de error para un conjunto de parámetros.

    Parameters:
        parameters (list): Parámetros del modelo.
        isotherm_model (func): Función que calcula valores predichos.
        data (dict): {'x': valores independientes, 'y': valores dependientes}.
        error_functions (dict): Diccionario de funciones de error.

    Returns:
        dict: Errores calculados para cada función.
    """
    y_pred = isotherm_model(data['x'], *parameters)
    return {name: func(data['y'], y_pred) for name, func in error_functions.items()}
    
def normalize_errors(errors, max_errors):
    """
    Normaliza los errores dividiéndolos por el máximo valor de cada función.

    Parameters:
        errors (dict): Errores calculados para cada función.
        max_errors (dict): Máximos valores de cada función.

    Returns:
        dict: Errores normalizados.
    """
    return {key: value / max_errors[key] for key, value in errors.items()}
    
def calculate_sne_scores(normalized_errors):
    """
    Calcula el puntaje SNE sumando errores normalizados.

    Parameters:
        normalized_errors (dict): Errores normalizados.

    Returns:
        float: Puntaje SNE.
    """
    return sum(normalized_errors.values())

    
def select_best_model(sne_scores):
    """
    Selecciona el mejor modelo basado en el puntaje SNE más bajo.

    Parameters:
        sne_scores (dict): Puntajes SNE para cada modelo.

    Returns:
        str: Nombre del mejor modelo.
    """
    return min(sne_scores, key=sne_scores.get)
import numpy as np
def main():

    data = {'x': np.array([0,0.0493, 0.0346, 0.0382, 0.0222, 0.0214, 0.0155, 0.0097, 0.0084, 0.0042]), 'y': np.array([0, 0.0251, 0.0249, 0.0253, 0.0234, 0.0240, 0.0180, 0.0160, 0.0154, 0.0070])}
    error_functions = {
        'RMSE': lambda y, y_pred: np.sqrt(np.mean((y - y_pred) ** 2)),
        'SSE': lambda y, y_pred: np.sum((y - y_pred) ** 2),
        'chi_squared': lambda y, y_pred: np.sum((y - y_pred) ** 2 / y_pred),
    }
    max_errors = {'RMSE': 1, 'SSE': 10, 'chi_squared': 5}  # Valores de ejemplo
    
    # Modelos isotérmicos
    models = {
        'Langmuir': lambda x, qmax, k: (qmax * k * x) / (1 + k * x),
        'Freundlich': lambda x, K, n: K * (x ** (1 / n)),
    }
    
    sne_scores = {}

    for model_name, isotherm_model in models.items():
        # Optimizar parámetros para una función de error
        initial_guess = [1, 1]  # Depende del modelo
        params = optimize_parameters(data, isotherm_model, error_functions['RMSE'], initial_guess)
        
        # Calcular errores para el modelo optimizado
        errors = calculate_all_errors(params, isotherm_model, data, error_functions)
        
        # Normalizar errores
        normalized = normalize_errors(errors, max_errors)
        
        # Calcular puntaje SNE
        sne_scores[model_name] = calculate_sne_scores(normalized)

    # Seleccionar mejor modelo
    best_model = select_best_model(sne_scores)
    print(f"El mejor modelo es: {best_model}")

import numpy as np
def main():

    data = {'x': np.array([0,0.0493, 0.0346, 0.0382, 0.0222, 0.0214, 0.0155, 0.0097, 0.0084, 0.0042]), 'y': np.array([0, 0.0251, 0.0249, 0.0253, 0.0234, 0.0240, 0.0180, 0.0160, 0.0154, 0.0070])}
    error_functions = {
        "SSE": lambda  y_exp, y_pred: sse(y_exp, y_pred),
        "SAE": lambda  y_exp, y_pred: sae(y_exp, y_pred),
        "ARE": lambda  y_exp, y_pred: are(y_exp, y_pred),
        "MPSD":lambda  y_exp, y_pred: mpsd(y_exp, y_pred),
        "chi_squared": lambda  y_exp, y_pred: chi_squared(y_exp, y_pred),
        "HYBRID": lambda  y_exp, y_pred: hybrid_value(y_exp, y_pred)
    }
    max_errors = {'RMSE': 1, 'SSE': 10, 'chi_squared': 5}  # Valores de ejemplo
    
    # Modelos isotérmicos
    models = {
        'Langmuir': lambda x, qmax, k: (qmax * k * x) / (1 + k * x),
        'Freundlich': lambda x, K, n: K * (x ** (1 / n)),
    }
    
    sne_scores = {}

    for model_name, isotherm_model in models.items():
        # Optimizar parámetros para una función de error
        initial_guess = [1, 1]  # Depende del modelo
        params = optimize_parameters(data, isotherm_model, error_functions['RMSE'], initial_guess)
        
        # Calcular errores para el modelo optimizado
        errors = calculate_all_errors(params, isotherm_model, data, error_functions)
        
        # Normalizar errores
        normalized = normalize_errors(errors, max_errors)
        
        # Calcular puntaje SNE
        sne_scores[model_name] = calculate_sne_scores(normalized)

    # Seleccionar mejor modelo
    best_model = select_best_model(sne_scores)
    print(f"El mejor modelo es: {best_model}")


In [20]:
import numpy as np
from scipy.optimize import minimize

# Definición de funciones de error
def sse(y_exp, y_pred):
    return np.sum((y_exp - y_pred) ** 2)

def sae(y_exp, y_pred):
    return np.sum(np.abs(y_exp - y_pred))

def are(y_exp, y_pred):
    return np.mean(np.abs((y_exp - y_pred) / y_exp)) * 100

def mpsd(y_exp, y_pred):
    return np.mean((y_exp - y_pred) ** 2 / (y_exp + y_pred)) * 100

def chi_squared(y_exp, y_pred):
    return np.sum((y_exp - y_pred) ** 2 / y_pred)

def hybrid_value(y_exp, y_pred):
    return np.sum(np.abs((y_exp - y_pred) / (y_exp + y_pred))) * 100

# Función para optimizar parámetros
def optimize_parameters(data, isotherm_model, error_function, initial_guess):
    def objective_function(params):
        y_pred = isotherm_model(data['x'], *params)
        return error_function(data['y'], y_pred)

    result = minimize(objective_function, initial_guess, method="Nelder-Mead")
    return result.x

# Función para calcular errores con todas las funciones de error
def calculate_all_errors(parameters, isotherm_model, data, error_functions):
    y_pred = isotherm_model(data['x'], *parameters)
    return {name: func(data['y'], y_pred) for name, func in error_functions.items()}

# Normalizar los errores
def normalize_errors(errors, max_errors):
    return {key: value / max_errors[key] for key, value in errors.items()}

# Calcular el puntaje SNE
def calculate_sne_scores(normalized_errors):
    return sum(normalized_errors.values())

# Seleccionar el mejor modelo
def select_best_model(sne_scores):
    return min(sne_scores, key=sne_scores.get)

# Función principal
def main():
    data = {'x': np.array([0,0.0493, 0.0346, 0.0382, 0.0222, 0.0214, 0.0155, 0.0097, 0.0084, 0.0042]), 
            'y': np.array([0, 0.0251, 0.0249, 0.0253, 0.0234, 0.0240, 0.0180, 0.0160, 0.0154, 0.0070])}
    
    # Definición de funciones de error
    error_functions = {
        "SSE": lambda y_exp, y_pred: sse(y_exp, y_pred),
        "SAE": lambda y_exp, y_pred: sae(y_exp, y_pred),
        "ARE": lambda y_exp, y_pred: are(y_exp, y_pred),
        "MPSD": lambda y_exp, y_pred: mpsd(y_exp, y_pred),
        "chi_squared": lambda y_exp, y_pred: chi_squared(y_exp, y_pred),
        "HYBRID": lambda y_exp, y_pred: hybrid_value(y_exp, y_pred)
    }
    
    max_errors = {'SSE': 10, 'SAE': 5, 'ARE': 15, 'MPSD': 20, 'chi_squared': 5, 'HYBRID': 30}
    
    # Modelos isotérmicos
    models = {
        'Langmuir': lambda x, qmax, k: (qmax * k * x) / (1 + k * x),
        'Freundlich': lambda x, K, n: K * (x ** (1 / n)),
    }
    
    sne_scores = {}

    # Optimización de parámetros y evaluación para cada modelo
    for model_name, isotherm_model in models.items():
        initial_guess = [1, 1]  # Depende del modelo
        params = optimize_parameters(data, isotherm_model, error_functions['SSE'], initial_guess)
        
        # Calcular errores para el modelo optimizado
        errors = calculate_all_errors(params, isotherm_model, data, error_functions)
        
        # Normalizar errores
        normalized = normalize_errors(errors, max_errors)
        
        # Calcular puntaje SNE
        sne_scores[model_name] = calculate_sne_scores(normalized)

    # Seleccionar el mejor modelo
    best_model = select_best_model(sne_scores)
    print(f"El mejor modelo es: {best_model}")

if __name__ == "__main__":
    main()


El mejor modelo es: Langmuir


/tmp/ipykernel_2398928/245956061.py:12: RuntimeWarning: invalid value encountered in divide
  return np.mean(np.abs((y_exp - y_pred) / y_exp)) * 100
/tmp/ipykernel_2398928/245956061.py:15: RuntimeWarning: invalid value encountered in divide
  return np.mean((y_exp - y_pred) ** 2 / (y_exp + y_pred)) * 100
/tmp/ipykernel_2398928/245956061.py:18: RuntimeWarning: invalid value encountered in divide
  return np.sum((y_exp - y_pred) ** 2 / y_pred)
/tmp/ipykernel_2398928/245956061.py:21: RuntimeWarning: invalid value encountered in divide
  return np.sum(np.abs((y_exp - y_pred) / (y_exp + y_pred))) * 100


In [21]:
main()

El mejor modelo es: Langmuir


/tmp/ipykernel_2398928/245956061.py:12: RuntimeWarning: invalid value encountered in divide
  return np.mean(np.abs((y_exp - y_pred) / y_exp)) * 100
/tmp/ipykernel_2398928/245956061.py:15: RuntimeWarning: invalid value encountered in divide
  return np.mean((y_exp - y_pred) ** 2 / (y_exp + y_pred)) * 100
/tmp/ipykernel_2398928/245956061.py:18: RuntimeWarning: invalid value encountered in divide
  return np.sum((y_exp - y_pred) ** 2 / y_pred)
/tmp/ipykernel_2398928/245956061.py:21: RuntimeWarning: invalid value encountered in divide
  return np.sum(np.abs((y_exp - y_pred) / (y_exp + y_pred))) * 100


In [10]:
import matplotlib.pyplot as plt
import numpy as np
from lmfit import Model
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Datos de muestra
ce = np.array([0,0.0493, 0.0346, 0.0382, 0.0222, 0.0214, 0.0155, 0.0097, 0.0084, 0.0042])
qe = np.array([0, 0.0251, 0.0249, 0.0253, 0.0234, 0.0240, 0.0180, 0.0160, 0.0154, 0.0070])

# Ordenar los datos
def order_sample(ce, qe):
    pears = sorted(zip(ce, qe))
    x, y= zip(*pears)
    return list(x), list(y)

# Función de Langmuir
def langmuir(ce, qm, k):
    return (qm * k * ce) / (1 + k * ce)

# Función de Freundlich
def freundlich(ce, kf, n):
    return kf * (ce ** (1 / n))

# Función para obtener los parámetros de Langmuir
def get_langmuir_params(ce, qm, k):
    model = Model(langmuir)
    params = model.make_params(qm=qm, k=k)
    result = model.fit(qe, params, ce=ce)
    return result.best_fit, result.aic, result.bic, len(params)

# Función para obtener los parámetros de Freundlich
def get_freundlich_params(ce, kf, n):
    model = Model(freundlich)
    params = model.make_params(kf=kf, n=n)
    result = model.fit(qe, params, ce=ce)
    return result.best_fit, result.aic, result.bic, len(params)

# Funciones de error
def sse(y_exp, y_pred):
    return np.sum((y_exp - y_pred) ** 2)

def sae(y_exp, y_pred):
    return np.sum(np.abs(y_exp - y_pred))

def are(y_exp, y_pred):
    return np.sum(np.abs((y_exp - y_pred) / y_exp))

def mpsd(y_exp, y_pred):
    # Asegurarse de que y_exp y y_pred son arrays de numpy
    y_exp = np.array(y_exp)
    y_pred = np.array(y_pred)

    # Cálculo de MPSD
    return np.sum((y_exp - y_pred) ** 2) / np.sum(y_exp ** 2)

def chi_squared(y_exp, y_pred):
    return np.sum(((y_exp - y_pred) ** 2) / y_pred)

def hybrid_value(y_exp, y_pred):
    return np.sum(np.abs((y_exp - y_pred) / y_pred))

# Calcular la SNE
def calculate_sne(y_exp, y_pred):
    sse_val = sse(y_exp, y_pred)
    sae_val = sae(y_exp, y_pred)
    are_val = are(y_exp, y_pred)
    mpsd_val = mpsd(y_exp, y_pred)
    chi_squared_val = chi_squared(y_exp, y_pred)
    hybrid_val = hybrid_value(y_exp, y_pred)
    
    errors = {
        "SSE": sse_val,
        "SAE": sae_val,
        "ARE": are_val,
        "MPSD": mpsd_val,
        "chi_squared": chi_squared_val,
        "HYBRID": hybrid_val
    }
    
    # Normalizar los errores
    normalized_errors = {key: value / max(errors.values()) for key, value in errors.items()}
    
    # Sumar todos los errores normalizados
    sne = np.sum(list(normalized_errors.values()))
    
    return sne

# Calcular los modelos
ce, qe = order_sample(ce, qe)

# Parámetros de Langmuir sacados de la linealización
qm = 0.0318
k = 96.7092
qe_langmuir_pred, lag_aic, lag_bic, lag_params_len = get_langmuir_params(ce, qm, k)

# Parámetros de Freundlich sacados de la linealización
kf = 0.1291
n = 2.0885
qe_freundlich_pred, f_aic, f_bic, f_params_len = get_freundlich_params(ce, kf, n)

# Calcular la SNE para ambos modelos
sne_langmuir = calculate_sne(qe, qe_langmuir_pred)
sne_freundlich = calculate_sne(qe, qe_freundlich_pred)

# Comparar SNE y elegir el mejor modelo
if sne_langmuir < sne_freundlich:
    best_model = "Langmuir"
else:
    best_model = "Freundlich"

# Imprimir el modelo con el mejor ajuste
print(f"El mejor modelo es: {best_model}")


El mejor modelo es: Freundlich


/tmp/ipykernel_2398928/3927130051.py:46: RuntimeWarning: invalid value encountered in divide
  return np.sum(np.abs((y_exp - y_pred) / y_exp))
/tmp/ipykernel_2398928/3927130051.py:57: RuntimeWarning: invalid value encountered in divide
  return np.sum(((y_exp - y_pred) ** 2) / y_pred)
/tmp/ipykernel_2398928/3927130051.py:60: RuntimeWarning: invalid value encountered in divide
  return np.sum(np.abs((y_exp - y_pred) / y_pred))
